In [4]:
#!/usr/bin/env python3
"""
Targeted download of SWOT L2 HR Raster 100m for a bbox and specific date windows.

Fix for your error:
- earthaccess can throw IndexError when the query returns 0 granules.
  We catch it and treat as empty (0 results).

Behavior:
- Only 100m
- You specify one or more DATE_WINDOWS
- Searches each window, auto-picks provider that returns the most 100m granules
- Filters by begin-time token in filename
- Downloads only missing files by default
"""

import os
import re
from datetime import datetime, timezone
import earthaccess

# ---------------------------
# USER CONFIG
# ---------------------------
DATA_DIR = "/Users/wasifshahriar/Documents/Masters research/Coding/swot_data"
BBOX = (89.8, 23.5, 92.8, 25.5)  # (lon_min, lat_min, lon_max, lat_max)

# Date windows you want (UTC dates). End date is inclusive.
DATE_WINDOWS = [
    ("2025-05-11", "2025-06-30"),
    # ("2024-06-01", "2024-06-30"),
]

# Providers to test
PROVIDERS = ["POCLOUD", "PODAAC"]

# This short_name worked for you before
DATASET_SHORT_NAME = "SWOT_L2_HR_Raster_D"

# Skip files already in folder (recommended)
SKIP_IF_EXISTS = True

# Optional: set Earthdata credentials via env vars here (if not already set)
os.environ["EARTHDATA_USERNAME"] = ""
os.environ["EARTHDATA_PASSWORD"] = ""

# ---------------------------
# FILENAME TIME PARSING
# ---------------------------
TS_RE = re.compile(r"_(20\d{6}T\d{6})_(20\d{6}T\d{6})_")  # _YYYYMMDDTHHMMSS_YYYYMMDDTHHMMSS_

def parse_start_time_from_name(name: str):
    """Parse begin timestamp from GranuleUR/filename token."""
    m = TS_RE.search(name)
    if not m:
        return None
    t0 = m.group(1)
    try:
        return datetime.strptime(t0, "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)
    except Exception:
        return None

def parse_date_utc(d: str):
    """Parse 'YYYY-MM-DD' -> UTC datetime at start of day."""
    return datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=timezone.utc)

def granule_ur(g):
    """earthaccess DataGranule -> GranuleUR string (usually filename)."""
    try:
        return g["umm"].get("GranuleUR")
    except Exception:
        return None

# ---------------------------
# SAFE earthaccess wrapper
# ---------------------------
def safe_search_data(**kwargs):
    """
    earthaccess sometimes crashes with IndexError when there are zero results.
    Treat that as empty results.
    """
    try:
        return earthaccess.search_data(count=-1, **kwargs)
    except IndexError:
        # This is the earthaccess "response[0]" bug when response=[]
        return []
    except Exception as e:
        # Other errors should still be visible
        raise

# ---------------------------
# PROVIDER SEARCH + FILTER
# ---------------------------
def search_granules_for_window_100m(date_start: str, date_end: str):
    """
    Search granules in bbox for the window and return the best-provider result list,
    already filtered to 100m by GranuleUR containing "_100m_".
    """
    token = "_100m_"

    best_provider = None
    best_filtered = []
    best_count = -1

    for prov in PROVIDERS + [None]:
        gs = safe_search_data(
            short_name=DATASET_SHORT_NAME,
            bounding_box=BBOX,
            temporal=(date_start, date_end),
            provider=prov,
        )

        filtered = []
        for g in gs:
            ur = granule_ur(g)
            if ur and token in ur.lower():
                filtered.append(g)

        print(f"[DIAG] Provider={prov} -> total={len(gs)} | 100m_filtered={len(filtered)} for {date_start}..{date_end}")

        if len(filtered) > best_count:
            best_count = len(filtered)
            best_provider = prov
            best_filtered = filtered

    print(f"[INFO] Chosen provider={best_provider} for {date_start}..{date_end} (100m n={len(best_filtered)})")
    return best_filtered

def filter_granules_by_begin_time(granules, date_start: str, date_end: str):
    """
    Keep granules whose begin time (from GranuleUR token) is within [start,end] inclusive.
    """
    t0 = parse_date_utc(date_start)
    t1 = parse_date_utc(date_end).replace(hour=23, minute=59, second=59)

    kept = []
    missing_ur = 0
    bad_time = 0

    for g in granules:
        ur = granule_ur(g)
        if not ur:
            missing_ur += 1
            continue
        bt = parse_start_time_from_name(ur)
        if bt is None:
            bad_time += 1
            continue
        if t0 <= bt <= t1:
            kept.append(g)

    return kept, missing_ur, bad_time

# ---------------------------
# DOWNLOAD
# ---------------------------
def download_specific_dates_100m():
    if not os.path.isdir(DATA_DIR):
        raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}")

    existing = set(os.listdir(DATA_DIR)) if SKIP_IF_EXISTS else set()

    # Login once
    earthaccess.login(persist=True)

    all_to_dl = []
    seen_ur = set()

    for (d0, d1) in DATE_WINDOWS:
        print("\n========================================")
        print(f"[INFO] Window: {d0} -> {d1} (UTC dates)")

        granules = search_granules_for_window_100m(d0, d1)

        if len(granules) == 0:
            print("[INFO] 0 granules returned for this window (likely no SWOT overpass in bbox).")
            continue

        kept, missing_ur, bad_time = filter_granules_by_begin_time(granules, d0, d1)
        print(f"[DIAG] Kept after begin-time filter: {len(kept)}")
        print(f"[DIAG] Missing GranuleUR: {missing_ur}")
        print(f"[DIAG] Unparseable begin-time token: {bad_time}")

        for g in kept:
            ur = granule_ur(g)
            if not ur:
                continue
            if SKIP_IF_EXISTS and (ur in existing or (ur + ".nc") in existing):
                continue
            if ur in seen_ur:
                continue
            seen_ur.add(ur)
            all_to_dl.append(g)

    print("\n========================================")
    print(f"[INFO] TOTAL granules queued for download (100m): {len(all_to_dl)}")

    if not all_to_dl:
        print("[INFO] Nothing to download for the specified windows.")
        print("      If you expected data, widen the window (e.g., +/- 10 days) because SWOT does not overpass daily.")
        return

    earthaccess.download(all_to_dl, local_path=DATA_DIR)
    print("[INFO] Download complete.")

if __name__ == "__main__":
    download_specific_dates_100m()


[INFO] Window: 2025-05-11 -> 2025-06-30 (UTC dates)
[DIAG] Provider=POCLOUD -> total=180 | 100m_filtered=90 for 2025-05-11..2025-06-30
[DIAG] Provider=PODAAC -> total=0 | 100m_filtered=0 for 2025-05-11..2025-06-30
[DIAG] Provider=None -> total=0 | 100m_filtered=0 for 2025-05-11..2025-06-30
[INFO] Chosen provider=POCLOUD for 2025-05-11..2025-06-30 (100m n=90)
[DIAG] Kept after begin-time filter: 90
[DIAG] Missing GranuleUR: 0
[DIAG] Unparseable begin-time token: 0

[INFO] TOTAL granules queued for download (100m): 90


QUEUEING TASKS | :   0%|          | 0/90 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/90 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/90 [00:00<?, ?it/s]

[INFO] Download complete.


In [19]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
SWOT HR Raster 100m: bbox + UTM-tile filename filtering (robust).

Why:
- CMR SpatialExtent metadata for HR Raster can be missing/too broad.
- earthaccess bbox searches may still return wrong tiles.
- Filename contains UTM tile (e.g., UTM46Q) which is a reliable geographic filter.

This script:
1) Searches a broad HR Raster dataset short_name that actually returns results.
2) Filters returned granules by GranuleUR pattern:
      - must contain "_Raster_100m_"
      - must contain one of UTM45Q/45R/46Q/46R (Bangladesh NE region)
3) Downloads only those.
"""

import os
import re
import glob
from datetime import datetime, timezone
import earthaccess
import xarray as xr
import numpy as np

# ---------------------------
# USER CONFIG
# ---------------------------

DATA_DIR = "/Users/wasifshahriar/Documents/Masters research/Coding/swot_data2"
os.makedirs(DATA_DIR, exist_ok=True)

BBOX = (89.8, 23.5, 92.8, 25.5)         # (lon_min, lat_min, lon_max, lat_max)
DATE_START = "2025-05-01"
DATE_END   = "2025-06-30"

# This one tends to return many results; we then filter by filename.
DATASET_SHORT_NAME = "SWOT_L2_HR_Raster_D"

# Providers to try; we will pick the one that returns results
PROVIDERS = ["POCLOUD", "PODAAC", None]

# Your expected UTM tiles for this bbox (Bangladesh NE: 45/46 and bands Q/R)
KEEP_UTM_TILES = ["UTM45Q", "UTM45R", "UTM46Q", "UTM46R"]

# ---------------------------
# HELPERS
# ---------------------------

def safe_search(**kwargs):
    """
    earthaccess sometimes crashes with IndexError when there are 0 hits.
    Treat that as 'no results' instead of hard-failing.
    """
    try:
        return earthaccess.search_data(**kwargs)
    except IndexError:
        return []
    except Exception as e:
        print("[WARN] search failed:", e)
        return []

def granule_ur(g):
    """
    Earthaccess returns dict-like granules in many setups.
    """
    try:
        return g["umm"].get("GranuleUR")
    except Exception:
        # fallback: some objects behave differently
        try:
            umm = getattr(g, "umm", {}) or {}
            return umm.get("GranuleUR")
        except Exception:
            return None

def keep_by_filename(gid: str) -> bool:
    if not gid:
        return False
    if "_Raster_100m_" not in gid:
        return False
    # must match one of the expected UTM tiles
    return any(f"_{t}_" in gid for t in KEEP_UTM_TILES)

def bbox_intersects(a, b):
    """a,b are (w,s,e,n)."""
    aw, as_, ae, an = a
    bw, bs, be, bn = b
    return not (ae < bw or aw > be or an < bs or as_ > bn)

def extent_from_file(nc_path):
    ds = xr.open_dataset(nc_path)
    lon = ds["longitude"].values
    lat = ds["latitude"].values
    ds.close()
    if isinstance(lon, np.ma.MaskedArray): lon = lon.filled(np.nan)
    if isinstance(lat, np.ma.MaskedArray): lat = lat.filled(np.nan)
    m = np.isfinite(lon) & np.isfinite(lat)
    if not m.any():
        return None
    return (float(lon[m].min()), float(lat[m].min()), float(lon[m].max()), float(lat[m].max()))

# ---------------------------
# MAIN
# ---------------------------

def main():
    print("[INFO] Target dir:", DATA_DIR)
    print("[INFO] BBOX:", BBOX)
    print("[INFO] Window (UTC dates):", DATE_START, "->", DATE_END)
    print("[INFO] short_name:", DATASET_SHORT_NAME)
    print("[INFO] Keep UTM tiles:", KEEP_UTM_TILES)

    # Login (set env vars beforehand):
    # export EARTHDATA_USERNAME="..."
    # export EARTHDATA_PASSWORD="..."
    earthaccess.login(strategy="environment", persist=True)

    # Try providers and choose the one that returns the most results
    best_provider = None
    best = []
    for prov in PROVIDERS:
        gs = safe_search(
            short_name=DATASET_SHORT_NAME,
            bounding_box=BBOX,
            temporal=(DATE_START, DATE_END),
            provider=prov,
            count=-1
        )
        print(f"[DIAG] provider={prov} -> returned {len(gs)} granules")
        if len(gs) > len(best):
            best = gs
            best_provider = prov

    if not best:
        raise RuntimeError("No granules returned by earthaccess for any provider in this window.")

    print(f"[INFO] Chosen provider={best_provider}, returned={len(best)}")

    # Filename/UTM filter (the key fix)
    kept = []
    for g in best:
        gid = granule_ur(g)
        if keep_by_filename(gid):
            kept.append(g)

    print(f"[INFO] After UTM+100m filename filter: kept={len(kept)} / {len(best)}")

    if not kept:
        print("[WARN] 0 kept. This likely means your bbox is correct but your expected UTM tiles list is incomplete.")
        print("       Next step: print first 20 GranuleURs and see which UTM codes appear.")
        for g in best[:20]:
            print("   ", granule_ur(g))
        return

    # De-dupe against existing local files
    existing = set(os.listdir(DATA_DIR))
    to_dl = []
    for g in kept:
        gid = granule_ur(g)
        if gid and (gid in existing or (gid + ".nc") in existing):
            continue
        to_dl.append(g)

    print(f"[INFO] New granules to download: {len(to_dl)}")
    if not to_dl:
        print("[INFO] Nothing to download.")
        return

    earthaccess.download(to_dl, local_path=DATA_DIR)
    print("[DONE] Download complete.")

    # Optional quick sanity check on first few downloaded files
    files = sorted(glob.glob(os.path.join(DATA_DIR, "*Raster_100m*.nc")))
    print(f"[DIAG] Local 100m files now: {len(files)}")
    for fn in files[:5]:
        ext = extent_from_file(fn)
        ok = (ext is not None and bbox_intersects(ext, BBOX))
        print(" ", os.path.basename(fn))
        print("   extent:", ext, " overlaps AOI?", ok)

if __name__ == "__main__":
    main()

[INFO] Target dir: /Users/wasifshahriar/Documents/Masters research/Coding/swot_data2
[INFO] BBOX: (89.8, 23.5, 92.8, 25.5)
[INFO] Window (UTC dates): 2025-05-01 -> 2025-06-30
[INFO] short_name: SWOT_L2_HR_Raster_D
[INFO] Keep UTM tiles: ['UTM45Q', 'UTM45R', 'UTM46Q', 'UTM46R']
[DIAG] provider=POCLOUD -> returned 218 granules
[DIAG] provider=PODAAC -> returned 0 granules
[DIAG] provider=None -> returned 0 granules
[INFO] Chosen provider=POCLOUD, returned=218
[INFO] After UTM+100m filename filter: kept=77 / 218
[INFO] New granules to download: 77


QUEUEING TASKS | :   0%|          | 0/77 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/77 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/77 [00:00<?, ?it/s]

[DONE] Download complete.
[DIAG] Local 100m files now: 77
  SWOT_L2_HR_Raster_100m_UTM45Q_N_x_x_x_032_258_057F_20250507T053148_20250507T053209_PID0_03.nc
   extent: (88.82316849219652, 22.769868951793253, 90.26797464909392, 24.103695587992583)  overlaps AOI? True
  SWOT_L2_HR_Raster_100m_UTM45Q_N_x_x_x_032_258_057F_20250507T053148_20250507T053209_PID0_04.nc
   extent: (88.82316849219652, 22.769868951793253, 90.26797464909392, 24.103695587992583)  overlaps AOI? True
  SWOT_L2_HR_Raster_100m_UTM45Q_N_x_x_x_033_258_057F_20250528T021654_20250528T021716_PID0_01.nc
   extent: (88.82224985873412, 22.769881445491343, 90.26884009621261, 24.103695587992583)  overlaps AOI? True
  SWOT_L2_HR_Raster_100m_UTM45Q_N_x_x_x_034_258_057F_20250617T230158_20250617T230219_PID0_01.nc
   extent: (88.82319381991475, 22.76905348841476, 90.26797464909392, 24.10367589951045)  overlaps AOI? True
  SWOT_L2_HR_Raster_100m_UTM45Q_N_x_x_x_034_258_057F_20250617T230158_20250617T230219_PID0_02.nc
   extent: (88.823193819